# SLAM Gauss-Newton Localization (Notebook)

A single reusable Gauss-Newton engine applied to four problems:

- **Q1 Single-Point Localization**: one robot at a fixed position; starting from a random/given initial guess, it iteratively recovers its own coordinates (1,2,1)
- **Q2 Trajectory**: one robot over 4 time steps, localized one step at a time and connected into a trajectory, with reverse validation
- **Q3 Multi-Robot (Swarm)**: 5 robots at the same time step; after localizing each, compute the 5x5 pairwise distance matrix
- **Q4 Integrated**: 5 robots x 200 time steps, with random landmarks and trajectories; solved using warm starts and visualized for trajectory and direction similarity

**On the rotation basis**: LiDAR returns a scalar distance (a vector norm), and an orthogonal basis change does not alter a norm, i.e. $\|B_L^{-1}v\|=\|v\|$. All problems are therefore expressed in the world frame ($B_w=I$), and $B_L$ is not handled separately.

> Every parameter that may need manual adjustment is annotated with the specific cell in which it lives.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
plt.rcParams["axes.unicode_minus"] = False
np.set_printoptions(precision=4, suppress=True)

ROBOT_COLORS = ["#2a9d8f","#e76f51","#e9c46a","#264653","#6a4c93"]
TEAL = LinearSegmentedColormap.from_list("teal", ["#eaf4f2","#0f4c5c"])

## Reusable Gauss-Newton Engine

Residual $r_i=h_i(X)-d_i$, where the predicted distance $h_i=\|c_i-X\|$; row $i$ of the Jacobian is $-(c_i-X)/h_i$.
One step returns R, J, JtJ, (JtJ)^-1, JtR, delta and X_next; SSR follows the instructor's definition $SSR=\frac12\sum r_i^2$.

In [ ]:
def gn_step(C, X, d):
    U = C - X
    h = np.linalg.norm(U, axis=1)
    R = h - d
    J = -U / h[:, None]
    JtJ = J.T @ J
    JtR = J.T @ R
    delta = -np.linalg.solve(JtJ, JtR)
    return {"R":R, "J":J, "JtJ":JtJ, "JtJinv":np.linalg.inv(JtJ),
            "JtR":JtR, "delta":delta, "Xnext":X+delta, "ssr":0.5*R@R}

def solve_unit(C, d, X0, max_iter=30, tol=1e-8):
    X = np.array(X0, float).copy()
    steps = []
    for _ in range(max_iter):
        st = gn_step(C, X, d)
        steps.append(st)
        X = st["Xnext"]
        if np.linalg.norm(st["delta"]) < tol:
            break
    converged = np.linalg.norm(steps[-1]["delta"]) < tol
    return {"finalX":X, "steps":steps, "converged":converged}

## Plotting Utilities (3D Trajectory + Heatmap)

In the 3D plots: circle = start, square = end, black dots = landmarks; the view can be rotated when run locally.

In [ ]:
def plot_tracks(tracks, landmarks=None, title=""):
    fig = plt.figure(figsize=(9,6))
    ax = fig.add_subplot(111, projection="3d")
    for i, tr in enumerate(tracks):
        tr = np.array(tr); c = ROBOT_COLORS[i % len(ROBOT_COLORS)]
        ax.plot(tr[:,0], tr[:,1], tr[:,2], color=c, lw=2)
        ax.scatter(*tr[0], color=c, marker="o", s=45)
        ax.scatter(*tr[-1], color=c, marker="s", s=45)
    if landmarks is not None:
        L = np.array(landmarks)
        ax.scatter(L[:,0], L[:,1], L[:,2], color="#111", s=35)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(title)
    return fig

def plot_heat(M, labels, title, annotate=True):
    fig, ax = plt.subplots(figsize=(6,5))
    M = np.array(M)
    im = ax.imshow(M, cmap=TEAL, vmin=np.min(M), vmax=np.max(M))
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    if annotate:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, format(M[i,j], ".3g"), ha="center", va="center",
                        color="#fff" if M[i,j] > (M.min()+M.max())/2 else "#16303a", fontsize=9)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    return fig

---
# Q1 Single-Point Localization

**To change the initial guess, landmarks or measured distances: edit only the cell below** (for example, set X0 to a random point with `np.random.uniform(-3,3,3)`).

In [ ]:
C1 = np.array([[4,0,0],
               [0,5,0],
               [0,0,6],
               [4,5,6]], float)
d1 = np.array([np.sqrt(14), np.sqrt(11), np.sqrt(30), np.sqrt(43)])
X0 = np.zeros(3)

q1 = solve_unit(C1, d1, X0)
print("Converged:", q1["converged"])
print("Final position X =", q1["finalX"])
for k, st in enumerate(q1["steps"]):
    print(f"Iter {k+1}  X={st['Xnext']}  SSR={round(st['ssr'],6)}")

### Q1 Full Matrices of the First Step (X[0] -> X[1])

In [ ]:
s0 = q1["steps"][0]
print("X[0] =", X0)
for key in ["R","J","JtJ","JtJinv","JtR","delta","Xnext"]:
    print(key, "=\n", s0[key], "\n")

### Q1 Reverse Validation

Substitute the final position back to compute the predicted distances and compare them with the measured ones; the residuals should all be close to zero.

In [ ]:
pred = np.linalg.norm(C1 - q1["finalX"], axis=1)
print("Predicted distances:", pred)
print("Measured distances:", d1)
print("Residuals:", pred - d1)

---
# Q2 Trajectory (One Robot x 4 Time Steps)

**To change the measured distances at the 4 time steps: edit only the cell below** (one row per time step, in the order A/B/C/D). The initial guess is fixed at zero and then warm-started along time.

In [ ]:
C2 = C1.copy()
d2 = np.array([
 [3.741657, 3.316625, 5.477226, 6.557439],
 [3.690528, 3.036445, 5.900847, 6.574192],
 [3.648287, 3.051229, 5.874521, 6.205643],
 [13.80507, 15.98687, 19.03103, 21.24100]])

track2, prev = [], np.zeros(3)
for t in range(len(d2)):
    u = solve_unit(C2, d2[t], prev)
    track2.append(u["finalX"]); prev = u["finalX"]
track2 = np.array(track2)
print(track2)

### Q2-2 Reverse Validation + Trajectory Plot

In [ ]:
for t in range(len(d2)):
    pred = np.linalg.norm(C2 - track2[t], axis=1)
    print(f"t{t} residuals:", np.round(pred - d2[t], 6))
plot_tracks([track2], C2, "Q2 Single-Robot Trajectory (circle=start, square=end)")
plt.show()

---
# Q3 Multi-Robot Localization (5 Robots x One Time Step)

**To change the landmarks or the 5 sets of measured distances: edit only the cell below**. Each robot is localized independently from zero (no time continuation at a single step), and the pairwise distance matrix is then computed.

In [ ]:
C3 = np.array([[-3,4,-7],
               [-21,14,0],
               [9,7,-18],
               [13,17,3]], float)
d3 = np.array([
 [18.41195, 35.58089, 24.18677, 20.24846],
 [8.944272, 18.574176, 23.874673, 31.064449],
 [7.81025, 24.37212, 13.37909, 16.43168],
 [9.273618, 20.712315, 25.019992, 28.757608],
 [20.34699, 31.92178, 29.76575, 19.20937]])

pos3 = np.array([solve_unit(C3, d3[i], np.zeros(3))["finalX"] for i in range(5)])
D3 = np.zeros((5,5))
for i in range(5):
    for j in range(5):
        D3[i,j] = np.linalg.norm(pos3[i]-pos3[j])
print("Robot positions:\n", pos3)
print("5x5 distance matrix:\n", D3)
plot_heat(D3, [f"R{i+1}" for i in range(5)], "Q3 Pairwise Robot Distance Matrix")
plt.show()

---
# Q4 Integrated (5 Robots x 200 Time Steps)

**To change the sampling size, time step or random seed: edit only T / dt / seed at the top of the cell below**.
Landmarks are generated along the four diagonal directions of a tetrahedron (guaranteeing they are non-coplanar and surround the scene), trajectories combine constant velocity with sinusoidal oscillation, and the solver uses warm starts.

In [ ]:
T = 200          # number of time steps
dt = 0.1         # time step length
seed = 42        # random seed (change this for a new batch)
rng = np.random.default_rng(seed)

tetra = np.array([[1,1,1],[1,-1,-1],[-1,1,-1],[-1,-1,1.]]) / np.sqrt(3)
C4 = np.array([np.round(d*rng.uniform(18,24) + rng.uniform(-1.5,1.5,3)) for d in tetra])

def gen_track(rng):
    p0 = rng.uniform(-9,9,3); v = rng.uniform(-0.7,0.7,3)
    A = rng.uniform(-3,3,3); w = rng.uniform(0.2,0.45); ph = rng.uniform(0,6.28)
    ts = np.arange(T)*dt
    return np.array([p0 + v*t + A*np.sin(w*t+ph) for t in ts])

true4 = [gen_track(rng) for _ in range(5)]
meas4 = [[np.linalg.norm(C4 - true4[i][k], axis=1) for k in range(T)] for i in range(5)]

solved4 = []
for i in range(5):
    tr, prev = [], np.zeros(3)
    for k in range(T):
        u = solve_unit(C4, meas4[i][k], prev)
        tr.append(u["finalX"]); prev = u["finalX"]
    solved4.append(np.array(tr))
print("Solved: 5 robots x", T, "time steps")

### Q4-1 Distance Matrix at a Selected Time Step

**To inspect other time steps: edit only pick_t in the cell below** (from 0 to T-1). Each time step corresponds to one 5x5 distance matrix.

In [ ]:
pick_t = 0
D4 = np.zeros((5,5))
for i in range(5):
    for j in range(5):
        D4[i,j] = np.linalg.norm(solved4[i][pick_t]-solved4[j][pick_t])
print(D4)
plot_heat(D4, [f"R{i+1}" for i in range(5)], f"t={round(pick_t*dt, 6)} Distance Matrix")
plt.show()

### Q4-2 True Trajectories vs. Solved Trajectories

Solid lines = true trajectories, dashed lines = solved trajectories; black dots = landmarks. Rotatable when run locally.

In [ ]:
fig = plt.figure(figsize=(10,7)); ax = fig.add_subplot(111, projection="3d")
for i in range(5):
    c = ROBOT_COLORS[i]
    ax.plot(true4[i][:,0], true4[i][:,1], true4[i][:,2], color=c, lw=2)
    ax.plot(solved4[i][:,0], solved4[i][:,1], solved4[i][:,2], color=c, lw=1, ls="--")
ax.scatter(C4[:,0], C4[:,1], C4[:,2], color="#111", s=35)
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
ax.set_title("Solid=true, dashed=solved")
plt.show()

### Q4-3 Per-Robot Trajectories (Faceted)

Each robot gets its own 3D view showing its full trajectory (circle = start, square = end), so they can be examined one at a time without occluding each other.

In [ ]:
fig = plt.figure(figsize=(13,8))
for i in range(5):
    ax = fig.add_subplot(2,3,i+1, projection="3d")
    tr = solved4[i]; c = ROBOT_COLORS[i]
    ax.plot(tr[:,0], tr[:,1], tr[:,2], color=c, lw=2)
    ax.scatter(*tr[0], color=c, marker="o", s=35)
    ax.scatter(*tr[-1], color=c, marker="s", s=35)
    ax.scatter(C4[:,0], C4[:,1], C4[:,2], color="#111", s=15)
    ax.set_title(f"R{i+1}"); ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
fig.tight_layout()
plt.show()

### Q4-4 Direction Similarity cos(theta)

For each trajectory, the cosine of the angle between the instantaneous velocity directions is averaged: closer to 1 means more aligned, closer to -1 means more opposed.
The heatmap uses shades of the same teal hue (on the absolute value) for coloring; read the true numbers in the cells for the sign.

In [ ]:
def cosine_matrix(tracks):
    n = len(tracks); vel = [np.diff(tr, axis=0) for tr in tracks]
    M = np.eye(n)
    for i in range(n):
        for j in range(i+1,n):
            vals = []
            for k in range(len(vel[i])):
                a, b = vel[i][k], vel[j][k]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 1e-9 and nb > 1e-9:
                    vals.append(a@b/(na*nb))
            M[i,j] = M[j,i] = np.mean(vals)
    return M

cosM = cosine_matrix(solved4)
labs = [f"R{i+1}" for i in range(5)]
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(np.abs(cosM), cmap=TEAL, vmin=0, vmax=1)
ax.set_xticks(range(5)); ax.set_xticklabels(labs)
ax.set_yticks(range(5)); ax.set_yticklabels(labs)
for i in range(5):
    for j in range(5):
        ax.text(j, i, format(cosM[i,j], ".3f"), ha="center", va="center",
                color="#fff" if abs(cosM[i,j])>0.55 else "#16303a", fontsize=9)
ax.set_title("cos(theta) Direction Similarity (shades of one hue)")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()
bi, bj = np.unravel_index(np.argmax(cosM - np.eye(5)), cosM.shape)
print(f"Most aligned trajectories: R{bi+1} and R{bj+1}, cos(theta)={cosM[bi,bj]:.4f}")

## Summary: Which Cell Holds Each Parameter

- Q1 initial guess / landmarks / measured distances: **Q1 input cell** (variables C1, d1, X0)
- Q2 measured distances at 4 steps: **Q2 data cell** (variable d2)
- Q3 landmarks / measured distances: **Q3 data cell** (variables C3, d3)
- Q4 size / step / random seed: **Q4 parameter cell** (variables T, dt, seed)
- Q4 distance matrix at other steps: **Q4-1 cell** (variable pick_t)
- All problems are computed in the world frame (Bw=I); at convergence SSR -> 0 and delta -> 0.